#### Model Training and Selection

This notebook compares different classification models and strategies for handling class imbalance. Model selection is performed using stratified cross-validation.

#### Data Loading

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [3]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

#### Load data

In [5]:
RANDOM_STATE = 42

project_root = Path.cwd()

if not (project_root / "data" / "processed" / "train.csv").exists():
    project_root = project_root.parent

train_path = project_root / "data" / "processed" / "train.csv"

if not train_path.exists():
    raise FileNotFoundError(f"Training data not found: {train_path}")

train_data = pd.read_csv(train_path)

print(f"Training data shape: {train_data.shape}")

Training data shape: (1087, 12)


In [6]:
X_train = train_data.drop(columns="quality_class")
y_train = train_data["quality_class"].astype(int)

#### Cross-Validation

In [7]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

#### Evaluation metrics

In [8]:
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "average_precision": "average_precision"
}

#### Baseline Model

Establish a reference using a classifier that always predicts the most frequent class.

In [9]:
baseline = DummyClassifier(strategy="most_frequent")

baseline_scores = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

In [10]:
def summarize_scores(model_name, strategy, scores):
    row = {
        "Model": model_name,
        "Strategy": strategy
    }

    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_std"] = values.std()

    return row

In [11]:
baseline_row = summarize_scores(
    "Dummy (most frequent)",
    "Baseline",
    baseline_scores
)

display(pd.DataFrame([baseline_row]).round(3))

,Model,Strategy,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,Dummy (most frequent),Baseline,0.865,0.002,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.135,0.002


The Dummy Classifier provides a baseline based only on the majority class. The other models should improve the detection of high-quality wines rather than relying on the dominant class.

#### Model and Strategy Comparison

Evaluate four classifiers using the original class distribution, class weighting and SMOTE.

#### Define models

In [12]:
base_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=1
    ),

    "SVM (RBF)": SVC(
        kernel="rbf",
        random_state=RANDOM_STATE
    )
}

#### Define strategies

In [13]:
strategies = [
    "Original",
    "Class weighting",
    "SMOTE"
]

results = [baseline_row]

In [14]:
def build_pipeline(base_model, strategy):
    model = clone(base_model)

    if strategy == "Class weighting":
        model.set_params(class_weight="balanced")

    if strategy == "SMOTE":
        return ImbPipeline([
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
            ("classifier", model)
        ])

    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", model)
    ])

All models use StandardScaler. SMOTE is applied after scaling and only within the training folds, preventing information leakage during cross-validation.

In [15]:
def evaluate_model(model_name, base_model, strategy):
    pipeline = build_pipeline(base_model, strategy)

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    return summarize_scores(model_name, strategy, scores)

#### Run comparisons

In [16]:
for model_name, base_model in base_models.items():
    for strategy in strategies:
        print(f"Evaluating {model_name} - {strategy}")

        row = evaluate_model(
            model_name,
            base_model,
            strategy
        )

        results.append(row)

Evaluating Logistic Regression - Original
Evaluating Logistic Regression - Class weighting
Evaluating Logistic Regression - SMOTE
Evaluating Decision Tree - Original
Evaluating Decision Tree - Class weighting
Evaluating Decision Tree - SMOTE
Evaluating Random Forest - Original
Evaluating Random Forest - Class weighting
Evaluating Random Forest - SMOTE
Evaluating SVM (RBF) - Original
Evaluating SVM (RBF) - Class weighting
Evaluating SVM (RBF) - SMOTE


In [17]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "average_precision_mean",
    ascending=False
).reset_index(drop=True)

display(results_df.round(3))

,Model,Strategy,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,Random Forest,SMOTE,0.860,0.016,0.732,0.033,0.487,0.052,0.557,0.067,0.519,0.050,0.871,0.024,0.538,0.071
1,Random Forest,Original,0.877,0.009,0.630,0.034,0.582,0.059,0.292,0.069,0.387,0.073,0.872,0.029,0.534,0.047
2,Random Forest,Class weighting,0.856,0.021,0.702,0.039,0.475,0.073,0.489,0.072,0.480,0.065,0.868,0.028,0.528,0.072
3,SVM (RBF),Class weighting,0.806,0.011,0.787,0.020,0.389,0.019,0.761,0.046,0.515,0.022,0.867,0.030,0.524,0.049
4,SVM (RBF),SMOTE,0.822,0.018,0.779,0.034,0.412,0.033,0.720,0.073,0.523,0.041,0.864,0.029,0.518,0.053
5,SVM (RBF),Original,0.873,0.013,0.591,0.028,0.596,0.123,0.204,0.050,0.302,0.068,0.814,0.050,0.508,0.078
6,Logistic Regression,SMOTE,0.785,0.027,0.787,0.023,0.367,0.031,0.789,0.051,0.499,0.029,0.855,0.033,0.494,0.086
7,Logistic Regression,Class weighting,0.786,0.023,0.799,0.015,0.371,0.023,0.816,0.047,0.508,0.017,0.856,0.032,0.493,0.091
8,Logistic Regression,Original,0.870,0.022,0.623,0.040,0.555,0.159,0.285,0.075,0.372,0.096,0.853,0.035,0.487,0.099
9,Decision Tree,SMOTE,0.813,0.030,0.676,0.047,0.364,0.070,0.489,0.078,0.416,0.072,0.676,0.047,0.252,0.048


The results show that handling class imbalance improves the detection of high-quality wines, although the effect varies across classifiers.

Logistic Regression with class weighting achieves the highest balanced accuracy (0.799) and recall (0.816), while SVM with SMOTE obtains the highest F1-score (0.523). Random Forest with SMOTE achieves the highest average precision (0.538).

No single configuration performs best across all metrics. Therefore, several candidates will be retained for further analysis and comparison before final model selection.

In [19]:
candidate_configs = [
    ("SVM (RBF)", "Class weighting"),
    ("SVM (RBF)", "SMOTE"),
    ("Logistic Regression", "Class weighting"),
    ("Random Forest", "SMOTE")
]

candidates_df = results_df[
    results_df[["Model", "Strategy"]]
    .apply(tuple, axis=1)
    .isin(candidate_configs)
].copy()

display(candidates_df.round(3))

,Model,Strategy,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,Random Forest,SMOTE,0.860,0.016,0.732,0.033,0.487,0.052,0.557,0.067,0.519,0.050,0.871,0.024,0.538,0.071
3,SVM (RBF),Class weighting,0.806,0.011,0.787,0.020,0.389,0.019,0.761,0.046,0.515,0.022,0.867,0.030,0.524,0.049
4,SVM (RBF),SMOTE,0.822,0.018,0.779,0.034,0.412,0.033,0.720,0.073,0.523,0.041,0.864,0.029,0.518,0.053
7,Logistic Regression,Class weighting,0.786,0.023,0.799,0.015,0.371,0.023,0.816,0.047,0.508,0.017,0.856,0.032,0.493,0.091


In [21]:
results_dir = project_root / "reports" / "tables"
results_dir.mkdir(parents=True, exist_ok=True)

candidates_path = results_dir / "candidate_models_cv.csv"

candidates_df.to_csv(candidates_path, index=False)

print(f"Candidate models saved to: {candidates_path}")

Candidate models saved to: C:\Users\amiro\University\4th\1st semester\IA\red-wine-quality-lab\reports\tables\candidate_models_cv.csv
